In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
    .master("local[*]")
    .appName("SparkOptimization")
    .config("spark.driver.memory", "2g")   
    .config("spark.sql.shuffle.partitions", 8)
    .getOrCreate())

print("cores:", spark.sparkContext.defaultParallelism)
print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

cores: 8
shuffle partitions: 8


In [2]:
weather = spark.read.parquet("data/weather_parquet")
print("weather partitions:", weather.rdd.getNumPartitions())

by_station = weather.groupBy("station").count()
print("after groupBy:", by_station.rdd.getNumPartitions())

spark.conf.set("spark.sql.shuffle.partitions", 200)
print(spark.conf.get("spark.sql.shuffle.partitions"))     # '200'

weather partitions: 8
after groupBy: 1
200


In [3]:
from datetime import datetime

big = spark.range(10_000_000).withColumn("fourth", F.col("id") ** 4)

for n in [1, 2, 4, 8, 16, 50, 200]:
    start = datetime.now()
    big.repartition(n).write.format("noop").mode("overwrite").save()
    print(n, "partitions:", (datetime.now() - start).total_seconds(), "sec")

1 partitions: 4.596998 sec
2 partitions: 17.105067 sec
4 partitions: 11.246029 sec
8 partitions: 7.358777 sec
16 partitions: 6.896167 sec
50 partitions: 7.505034 sec
200 partitions: 12.766955 sec


In [3]:
from datetime import datetime

big = spark.range(10_000_000).withColumn("fourth", F.col("id") ** 4)

for n in [1, 2, 4, 8, 16, 50, 200]:
    start = datetime.now()
    big.repartition(n).write.format("noop").mode("overwrite").save()
    print(n, "partitions:", (datetime.now() - start).total_seconds(), "sec")

1 partitions: 2.399181 sec
2 partitions: 9.343532 sec
4 partitions: 7.290971 sec
8 partitions: 5.683435 sec
16 partitions: 4.193442 sec
50 partitions: 5.029857 sec
200 partitions: 9.394825 sec


In [4]:
weather = spark.read.parquet("data/weather_parquet")

summary = (weather
    .where(F.col("temp") > 15)
    .withColumn("temp_f", F.col("temp") * 9/5 + 32)
    .groupBy("station", "county")
    .agg(F.avg("temp_f").alias("avg_temp_f")))

In [5]:
%%time
summary.count()

CPU times: user 1.07 ms, sys: 269 μs, total: 1.33 ms
Wall time: 1.06 s


25

In [7]:
summary.cache()

DataFrame[station: string, county: string, avg_temp_f: double]

In [8]:
%%time
summary.count()

CPU times: user 965 μs, sys: 3.78 ms, total: 4.75 ms
Wall time: 3.9 s


25

In [9]:
%%time
summary.count()

CPU times: user 0 ns, sys: 3.67 ms, total: 3.67 ms
Wall time: 643 ms


25

In [10]:
%%time
summary.count()

CPU times: user 2.71 ms, sys: 674 μs, total: 3.38 ms
Wall time: 573 ms


25

In [11]:
print(summary.is_cached)       # True
print(summary.storageLevel)    # Disk Memory Deserialized 1x Replicated

summary.unpersist()
print(summary.is_cached)       # False

True
Disk Memory Deserialized 1x Replicated
False


#### practicing Errors, Debugging and Troubleshooting

In [6]:
weather.select("temperature").show() 

{"ts": "2026-10-15 23:34:06.640", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `temperature` cannot be resolved. Did you mean one of the following? [`temp`, `date`, `dewpt`, `latitude`, `longitude`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.DirectMethodHandleAccessor.invoke(DirectMethodHandleAccessor.java", "line": "103)", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o88.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `temperature` cannot be resolved. Did you mean one of the following? [`temp`, `date`, `dewpt`, `latitude`, `longitude`]. SQLSTATE: 42703;\n'Project ['temperature]\n+- Relation [county#43,station#44,latitude#45,longitude#46,date#47,rain#48,tem

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `temperature` cannot be resolved. Did you mean one of the following? [`temp`, `date`, `dewpt`, `latitude`, `longitude`]. SQLSTATE: 42703;
'Project ['temperature]
+- Relation [county#43,station#44,latitude#45,longitude#46,date#47,rain#48,temp#49,wetb#50,dewpt#51,vappr#52,rhum#53,msl#54,wdsp#55,wddir#56,sun#57,vis#58,clht#59,clamt#60] parquet


In [7]:
for t in weather["temp"]: print(t)

PySparkTypeError: [NOT_ITERABLE] Column is not iterable.

In [8]:
weather.map(lambda r: r["temp"])

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `map` is not supported.

In [9]:
result = weather.groupBy("station").count().show()
result.orderBy("count")

+---------------+------+
|        station| count|
+---------------+------+
|  KNOCK AIRPORT|208938|
|         FINNER|193411|
|        GURTEEN|108144|
|    JOHNSTOWNII|135192|
|      BELMULLET|266617|
|       CASEMENT|266617|
|        ATHENRY| 78312|
|     BALLYHAISE|133704|
|       OAK PARK|147576|
|SHANNON AIRPORT|266617|
|   PHOENIX PARK|120144|
|   ROCHES POINT|247296|
|    CLAREMORRIS|266580|
|   CORK AIRPORT|266617|
|        NEWPORT|135144|
|      MULLINGAR|266617|
|      MT DILLON|108192|
|     MOORE PARK|147576|
|      MACE HEAD|138480|
|     MALIN HEAD|266617|
+---------------+------+
only showing top 20 rows


AttributeError: 'NoneType' object has no attribute 'orderBy'

In [10]:
weather.where(F.col("temp") > 15 and F.col("rain") > 0).show()

PySparkValueError: [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions.

In [11]:
weather.select("station", "temp").where(F.col("temp") > 15).explain()

== Physical Plan ==
*(1) Filter (isnotnull(temp#49) AND (temp#49 > 15.0))
+- *(1) ColumnarToRow
   +- FileScan parquet [station#44,temp#49] Batched: true, DataFilters: [isnotnull(temp#49), (temp#49 > 15.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/weather_parquet], PartitionFilters: [], PushedFilters: [IsNotNull(temp), GreaterThan(temp,15.0)], ReadSchema: struct<station:string,temp:double>


